In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib
from pathlib import Path



# 1. Chargement
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
df = pd.read_csv(url)

# 2. Séparation de la cible et nettoyage des fuites de données
target = "Machine failure"
leakage_cols = ["UDI", "Product ID", "TWF", "HDF", "PWF", "OSF", "RNF"]
df_clean = df.drop(columns=leakage_cols)

# 3. Feature Engineering physique
df_clean["Power [W]"] = df_clean["Torque [Nm]"] * (df_clean["Rotational speed [rpm]"] * (2 * np.pi / 60))
df_clean["Temp_diff [K]"] = df_clean["Process temperature [K]"] - df_clean["Air temperature [K]"]
df_clean["Wear_Torque_interaction"] = df_clean["Tool wear [min]"] * df_clean["Torque [Nm]"]

# Encodage One-Hot de la colonne Type (L, M, H)
df_clean = pd.get_dummies(df_clean, columns=["Type"], drop_first=True)

X = df_clean.drop(columns=[target])
y = df_clean[target]

# 4. Découpage stratifié Train / Test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 5. Modélisation avec pénalisation du déséquilibre
clf = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    class_weight="balanced",  # Force l'algorithme à sur-pénaliser les pannes manquées
    random_state=42
)
clf.fit(X_train, y_train)

# 6. Évaluation
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print("=== Matrice de confusion ===")
print(confusion_matrix(y_test, y_pred))
print("\n=== Rapport de classification ===")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score : {roc_auc_score(y_test, y_proba):.4f}")

# 7. Sauvegarde du modèle


# Définition du chemin dans ton dossier personnel
save_path = Path.home() / "model.pkl"

# Sauvegarde explicite avec la variable save_path
joblib.dump(clf, str(save_path))
print(f"Modèle sauvegardé avec succès dans : {save_path}")

=== Matrice de confusion ===
[[1928    4]
 [  15   53]]

=== Rapport de classification ===
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      1932
           1       0.93      0.78      0.85        68

    accuracy                           0.99      2000
   macro avg       0.96      0.89      0.92      2000
weighted avg       0.99      0.99      0.99      2000

ROC-AUC Score : 0.9798
Modèle sauvegardé avec succès dans : /Users/yassoimarcalexdoh/model.pkl
